# 결정 규칙

이 규칙을 쓰기 전에는 그룹을 나누지 않은 값만 봤다. 행 수, 결측, 중복, 전체 리텐션, 전체 플레이 수 분포, 감지 가능한 최소 차이(MDE)다.

## 1. 질문과 행동

**질문.** 첫 관문을 레벨 30에서 40으로 옮기면 7일 리텐션이 옮길 가치가 있을 만큼 오르는가.

**기본값은 "그대로 둔다"이다.** 옮기는 데 비용이 들고 리텐션 외의 이득(매출 등)은 이 데이터로 알 수 없다. 그래서 리텐션이 1%p 이상 오르는 것이 확실해야만 옮긴다.

차이는 `gate_40 − gate_30`으로 계산한다.

| 7일 리텐션 차이의 95% 신뢰구간 | 권고 | 서술 |
|---|---|---|
| 전체가 +1%p보다 위 | 40으로 옮긴다 | 옮길 가치가 있을 만큼 오른다 |
| 전체가 0보다 위지만 +1%p에 못 미치거나 걸침 | 그대로 둔다 | 오르지만 옮길 만큼 큰지 확실하지 않다 |
| 0을 포함 | 그대로 둔다 | 오른다는 증거를 찾지 못했다 |
| 전체가 0보다 아래 | 그대로 둔다 | 옮기면 리텐션이 떨어진다 |

이 권고는 리텐션 기준이다. 매출은 이 데이터로 판단할 수 없다.

## 2. 주 지표

**`retention_7`** (설치 후 정확히 7일째에 플레이했는가. "7일 안에"가 아니다.)

처치가 지표에 닿아야 한다. 전체 사용자의 63%는 14일 동안 30판도 하지 않아 레벨 30 관문을 만나지 않았고 레벨 30~40에 가려면 7일 동안 하루 평균 4~6레벨을 올려야 하므로 1일째에는 관문에 도달한 사용자가 거의 없다고 가정한다. 이 가정은 레벨 도달 시점 데이터가 없어 확인할 수 없다.

## 3. 보조 지표

결정에는 쓰지 않고 주 지표 결과를 이해하는 데 쓴다.

- **`retention_1`**: 점검용. 위 가정이 맞다면 두 그룹에서 차이가 없어야 한다. 차이가 나면 가정이 틀렸거나 배정에 문제가 있다는 신호로 본다.
- **`sum_gamerounds`**: 리텐션이 변할 때 플레이 양도 함께 변했는지 본다.

## 4. 검정 방식

- 양측 검정. 관문을 옮기면 리텐션이 오를 수도 내릴 수도 있다.
- 주 지표: 유의수준 0.05. 두 비율 z검정과 차이의 95% 신뢰구간.
- 보조 지표 2개: 본페로니 보정으로 각각 유의수준 0.025.

## 5. 판단 기준

**1%p.** 사업 맥락이 없으므로 가정이다.

- 뜻: 옮길 가치가 있는 최소 상승폭.
- 근거: 7일째에 돌아오는 사용자는 100명 중 18.6명이다. 1%p는 그중 약 1명, 5% 정도다. 이만큼은 늘어야 옮기는 비용을 감수할 만하다고 가정한다.
- MDE(0.73%p)보다 크므로 이 실험으로 가를 수 있는 크기다.

차이의 크기를 서술할 때도 같은 기준을 쓴다.

| 신뢰구간의 위치 | 크기에 대한 서술 |
|---|---|
| 전체가 −1%p와 +1%p 사이 | 차이가 있어도 작다 |
| 전체가 그 바깥 | 차이가 크다 |
| 1%p 선을 걸침 | 크기는 불확실하다 |

## 6. 표본 비율 점검 (SRM)

- 카이제곱 적합도 검정으로 그룹 인원이 50:50에 맞는지 본다.
- 기준: p값이 0.01보다 작으면 SRM이 의심된다고 본다.
- 걸리면: 분석은 계속하되 결론의 확신을 낮추고 한계에 적는다. 공개 데이터라 배정 과정을 조사할 수 없기 때문이다.
- 통과하더라도 배정 과정을 확인할 수 없다는 점을 한계에 적는다.

## 7. 데이터 처리

- 사용자를 걸러내지 않는다. 90,189명 전원을 분석한다. 플레이 수는 관문 위치의 영향을 받는 값이라, 그것으로 사용자를 빼면 두 그룹을 비교할 수 없게 된다.
- 극단값 사용자도 빼지 않는다. 플레이 수는 평균 대신 중앙값으로 비교한다.


## 결정 규칙 전에 본 것

아래 셀은 모두 그룹(`version`)을 나누지 않고 계산한다.

### 데이터 불러오기

In [9]:
import numpy as np
import pandas as pd
from scipy.stats import norm

df = pd.read_csv('data/cookie_cats.csv')
print(df.shape)
df.head()

(90189, 5)


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


### 결측과 중복

In [10]:
print("결측:", df.isna().sum().sum())
print("중복 userid:", df["userid"].duplicated().sum())

결측: 0
중복 userid: 0


### 전체 리텐션

In [11]:
ret1_all = df["retention_1"].mean()
ret7_all = df["retention_7"].mean()

print(f"전체 1일 리텐션: {ret1_all:.1%}")
print(f"전체 7일 리텐션: {ret7_all:.1%}")

전체 1일 리텐션: 44.5%
전체 7일 리텐션: 18.6%


### 감지 가능한 최소 차이 (MDE)

유의수준 5%(양측), 검정력 80% 기준. 그룹별 실제 인원은 아직 보지 않았으므로 반반이라고 가정한다.

In [12]:
n_per_group = len(df) / 2


def mde(p, n):
    se = np.sqrt(2 * p * (1 - p) / n)
    return (1.96 + 0.84) * se


print(f"MDE (1일 리텐션): {mde(ret1_all, n_per_group):.2%}p")
print(f"MDE (7일 리텐션): {mde(ret7_all, n_per_group):.2%}p")

MDE (1일 리텐션): 0.93%p
MDE (7일 리텐션): 0.73%p


### 전체 플레이 수 분포

In [13]:
rounds = df["sum_gamerounds"]

print(f"평균: {rounds.mean():.1f}")
print(f"중앙값: {rounds.median():.0f}")
print(f"99% 분위수: {rounds.quantile(0.99):.0f}")
print(f"최댓값: {rounds.max()}")
print(f"0판 사용자: {(rounds == 0).mean():.1%}")
print(f"30판 미만 사용자: {(rounds < 30).mean():.1%}")

평균: 51.9
중앙값: 16
99% 분위수: 493
최댓값: 49854
0판 사용자: 4.4%
30판 미만 사용자: 63.1%
